# DAE-PINN

## 环境安装

This example requires **MindSpore >= 2.5.0** to call the following interfaces: *mindspore.jit, mindspore.jit_class, mindspore.data_sink*. For details, please refer to [MindSpore Installation](https://www.mindspore.cn/install).

In addition, **MindScience >=0.1.0** is required. If it is not installed in your current environment, please install it according to the following steps, selecting the backend and version.


In [ ]:
mindscience_version = "0.1.0"  # update if needed
# Comment out the following code if you are using NPU.
!pip uninstall -y mindscience-ascend
!pip install mindscience-ascend==$mindscience_version

# NPU Uncomment if needed.
# !pip uninstall -y mindscience-ascend
# !pip install mindscience-ascend==$mindscience_version

## 概述

- **电力网络动态安全评估需求** ：随着电力网络中分布式能源资源的整合、市场自由化以及复杂通信和控制算法的采用，电力网络的运行条件和潜在故障场景变得更加多样化，影响其安全性。为了评估电力网络的动态安全性，需要模拟其在面对单一故障时的动态响应，这需要求解一组非线性微分代数方程（DAE），而传统显式积分方案在求解 DAE 时会失败，商业求解器计算成本高、内存需求大，限制了动态安全评估的在线部署。

- **深度学习在科学和工程领域的潜力与挑战** ：尽管深度学习在计算机视觉和自然语言处理等领域取得了巨大成功，但在学习科学和工程动态系统方面应用有限，因为数据收集成本高昂，且大多数传统深度学习方法在数据量有限的情况下缺乏鲁棒性和泛化能力。

## 工作原理

* DAE-PINNs 框架结合了隐式龙格 - 库塔时间步进方案（专为求解 DAE 设计）和物理信息神经网络（PINN）。在时间步进过程中，假设已积分至 $(t_n, y_n, z_n)$，目标是推进至 $(t_{n+1}, y_{n+1}, z_{n+1})$，应用隐式龙格 - 库塔方案后，得到一系列方程，包括内部阶段的更新公式和最终状态的更新公式。

* 通过惩罚方法强制神经网络满足 DAE 作为近似硬约束。在训练过程中，将 DAE 的残差作为损失函数的一部分，使得网络在学习过程中不仅拟合数据，还能满足物理定律所描述的 DAE 方程，从而将物理信息融入到神经网络的学习过程中。

## 方法细节

* **问题设置** ：DAE 以半显式形式给出，包括动态状态 y 和代数变量 z，以及描述微分方程的 f 和代数方程的 g。假设 f 和 g 具有足够高的可微性，并且 DAE 的索引为 1，即雅可比矩阵 g_z 的逆存在且在精确解附近有界，这使得代数方程在局部有唯一解 $z = G(y)$，从而 DAE 可以转化为普通微分方程系统。

* **网络结构** ：与标准的 PINN 类似，DAE-PINNs 通常由输入层、多个隐藏层和输出层组成。输入层接收时间和动态状态等信息，隐藏层通过非线性激活函数进行特征提取和转换，输出层预测代数变量的值。

* **损失函数** ：损失函数由两部分组成，一部分是数据损失，用于拟合初始条件、边界条件等已知点的数据；另一部分是物理损失，即 DAE 的残差损失，通过自动微分计算网络输出对时间和状态变量的导数，代入 DAE 方程得到残差，并将其作为物理损失的一部分。通过优化这两个部分的损失函数，使得网络既能拟合数据，又能满足物理方程。
![model](./images/model.png)
[DAE-PINN](https://arxiv.org/abs/2109.04304)的网络结构如上图。
与传统神经网络不同，DAE-PINN 在网络结构中融入了物理信息，通过构造特定的损失函数，使网络在学习过程中不仅拟合数据，还能满足物理定律所描述的 DAE 方程，从而提高了模型的准确性和泛化能力。整体的网络架构如上，分为两个网络分别处理动态状态和代数状态，网络的输入是包括时间信息和动态状态信息，网络的输出是对动态状态和代数状态的预测值，具体来说，会输出动态状态 y 和代数变量 z 的预测结果，如在电力网络案例中，输出动态状态和代数变量的预测以实现对电力网络动态行为的模拟。网络支持使用`fnn`、`attention`、`conv1d`3种backbone。`fnn`为多层感知机网络，`attention`为采用类似transformer attention形式的FFN网络，`conv1d`为使用了`Conv1D`的FFN网络。

## 准备环节

实践前，确保已经正确安装最新版本的MindSpore与mindscience。如果没有，可以通过：

* [MindSpore安装页面](https://www.mindspore.cn/install) 安装MindSpore。

* [mindscience安装页面](https://gitee.com/mindspore/mindscience) 安装mindscience。


## Overview

* **Dynamic Security Assessment Needs of Power Networks**: With the integration of distributed energy resources into power networks, market liberalization, and the adoption of complex communication and control algorithms, the operating conditions and potential fault scenarios of power networks are becoming more diverse, affecting their security. To evaluate the dynamic security of power networks, it is necessary to simulate their dynamic response when facing a single fault. This requires solving a set of nonlinear differential-algebraic equations (DAEs). Traditional explicit integration schemes fail to solve DAEs, and commercial solvers are computationally expensive and memory-intensive, limiting the online deployment of dynamic security assessment.
* **The Potential and Challenges of Deep Learning in Scientific and Engineering Fields**: Despite the significant success of deep learning in computer vision and natural language processing, its application in learning scientific and engineering dynamic systems is limited. This is due to the high cost of data collection and the lack of robustness and generalization ability of most traditional deep learning methods when data is limited.

## How It Works

* The DAE-PINNs framework combines an implicit Runge-Kutta time-stepping scheme (designed specifically for solving DAEs) with physics-informed neural networks (PINNs). During the time-stepping process, assuming integration has reached $(t_n, y_n, z_n)$, the goal is to advance to $(t_{n+1}, y_{n+1}, z_{n+1})$. After applying the implicit Runge-Kutta scheme, a series of equations are obtained, including update formulas for internal stages and final states.
* The neural network is constrained to satisfy the DAE through a penalty method. During training, the residual of the DAE is used as part of the loss function. This enables the network to not only fit the data but also satisfy the DAE equations described by physical laws during the learning process, thereby incorporating physical information into the learning process of the neural network.

## Method Details

* **Problem Setup**: The DAE is given in semi-explicit form, including dynamic states y and algebraic variables z, as well as f describing the differential equations and g describing the algebraic equations. It is assumed that f and g are sufficiently differentiable and that the DAE has an index of 1, i.e., the Jacobian matrix g_z is invertible and bounded near the exact solution. This ensures that the algebraic equations have a unique local solution $z = G(y)$, allowing the DAE to be transformed into a system of ordinary differential equations.
* **Network Structure**: Similar to standard PINNs, DAE-PINNs typically consist of an input layer, multiple hidden layers, and an output layer. The input layer receives information such as time and dynamic states, the hidden layers extract and transform features through nonlinear activation functions, and the output layer predicts the values of algebraic variables.
* **Loss Function**: The loss function consists of two parts. One part is the data loss, used to fit data from known points such as initial and boundary conditions. The other part is the physics loss, which is the residual loss of the DAE. By automatically differentiating the network's outputs with respect to time and state variables, the residual of the DAE equation is obtained and used as part of the physics loss. Optimizing these two parts of the loss function enables the network to both fit the data and satisfy the physical equations.

![model](./images/model.png)

As shown in the figure above, the network structure of DAE-PINN differs from traditional neural networks in that it incorporates physical information. By constructing a specific loss function, the network not only fits the data during the learning process but also satisfies the DAE equations described by physical laws. This enhances the accuracy and generalization ability of the model. The overall network architecture includes two networks that separately process dynamic states and algebraic states. The network inputs include time information and dynamic state information, and the outputs are predictions of dynamic states and algebraic variables. For example, in the case of a power network, the network outputs predictions of dynamic states and algebraic variables to simulate the dynamic behavior of the power network. The network supports three types of backbones: `fnn`, `attention`, and `conv1d`. `fnn` is a multi-layer perceptron network, `attention` is a FFN network adopting a transformer-like attention mechanism, and `conv1d` is a FFN network utilizing `Conv1D`.

## Preparation

Before practice, ensure that the latest versions of MindSpore and mindscience are correctly installed. If not, you can install them through:

* [MindSpore Installation Page](https://www.mindspore.cn/install)
* [mindscience Installation Page](https://gitee.com/mindspore/mindscience)

## DAE-PINN Implementation

The implementation of DAE-PINN is divided into the following 5 steps:

  1. Configure network and training parameters
  2. Dataset creation and loading
  3. Model construction
  4. Model training
  5. Result visualization


In [ ]:
import time

from mindspore import ops, jit
from mindspore import context
from mindspore.experimental import optim
import numpy as np

from mindscience.utils import load_yaml_config

from src.utils import dotdict
from src.model import three_bus_PN
from src.data import get_dataset
from src.trainer import DaeTrainer


In [ ]:
context.set_context(mode=context.PYNATIVE_MODE, device_target='Ascend', device_id=1)
config = load_yaml_config('./configs/config.yaml')
model_params, data_params, optim_params, ode_params, summary_params = config[
    'model'], config['data'], config['optimizer'], config['ode'], config['summary']

dynamic = dotdict()
dynamic.num_IRK_stages = model_params['num_IRK_stages']
dynamic.state_dim = 4
dynamic.activation = model_params['dyn_activation']
dynamic.initializer = "Glorot normal"
dynamic.dropout_rate = 0
dynamic.batch_normalization = None if model_params['dyn_bn'] == "no-bn" else model_params['dyn_bn']
dynamic.layer_normalization = None if model_params['dyn_ln'] == "no-ln" else model_params['dyn_ln']
dynamic.type = model_params['dyn_type']

if model_params['unstacked']:
    dim_out = dynamic.state_dim * (dynamic.num_IRK_stages + 1)
else:
    dim_out = dynamic.num_IRK_stages + 1

if model_params['use_input_layer']:
    dynamic.layer_size = [dynamic.state_dim * 5] + \
        [model_params['dyn_width']] * model_params['dyn_depth'] + [dim_out]
else:
    dynamic.layer_size = [dynamic.state_dim] + \
        [model_params['dyn_width']] * model_params['dyn_depth'] + [dim_out]

algebraic = dotdict()
algebraic.num_IRK_stages = model_params['num_IRK_stages']
dim_out_alg = algebraic.num_IRK_stages + 1
algebraic.layer_size = [dynamic.state_dim] + \
    [model_params['alg_width']] * model_params['alg_depth'] + [dim_out_alg]
algebraic.activation = model_params['alg_activation']
algebraic.initializer = "Glorot normal"
algebraic.dropout_rate = 0
algebraic.batch_normalization = None if model_params['alg_bn'] == "no-bn" else model_params['alg_bn']
algebraic.layer_normalization = None if model_params['alg_ln'] == "no-ln" else model_params['alg_ln']
algebraic.type = model_params['alg_type']

## Dataset Creation and Loading

Dataset download link:
This file contains 6000 samples of the HyperCube dataset.


In [ ]:
train_dataset, test_dataset, val_dataset = get_dataset(data_params)

## Model construction


In [ ]:

net = three_bus_PN(
    dynamic,
    algebraic,
    use_input_layer=model_params['use_input_layer'],
    stacked=not model_params['unstacked'],
)

## Loss Function and Optimizer

The loss function uses the MSE function. The optimizer selects Adam, and the learning rate scheduling uses ReduceLROnPlateau.


In [ ]:
num_IRK_stages = model_params['num_IRK_stages']
# collecting RK data
data_dir = data_params['data_dir']
irk_data = np.loadtxt(os.path.join(data_dir, 'IRK_weights', f'Butcher_IRK{num_IRK_stages}.txt'), ndmin=2, dtype=np.float32)
IRK_weights = np.reshape(irk_data[0:num_IRK_stages**2+num_IRK_stages], (num_IRK_stages+1, num_IRK_stages))
IRK_weights = Tensor(IRK_weights)
IRK_times = irk_data[num_IRK_stages**2 + num_IRK_stages:]
trainer = DaeTrainer(net, IRK_weights=IRK_weights, IRK_times=IRK_times, h=ode_params['h'],
                     dyn_weight=model_params['dyn_weight'], alg_weight=model_params['alg_weight'])

optimizer = optim.Adam(net.trainable_params(), lr=optim_params['lr'])
scheduler_type = optim_params['scheduler_type']
use_scheduler = optim_params['use_scheduler']
if use_scheduler:
    if scheduler_type == "plateau":
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            patience=optim_params['patience'],
            factor=optim_params['factor'],
        )
    elif scheduler_type == "step":
        scheduler = optim.lr_scheduler.StepLR(
            optimizer, step_size=optim_params['patience'], gamma=optim_params['factor']
        )
    else:
        scheduler = None
else:
    scheduler = None


## Training Function

Using MindSpore >= 2.5.0, you can train neural networks in a functional programming paradigm. The single-step training function is decorated with jit.


In [ ]:

def forward_fn(x):
    loss = trainer.get_loss(x)
    return loss

grad_fn = ops.value_and_grad(
    forward_fn, None, optimizer.parameters, has_aux=False)

@jit
def train_step(x):
    loss, grads = grad_fn(x)
    optimizer(grads)
    return loss

## Model Training

During model training, training and inference are performed simultaneously. Users can directly load the test dataset and output the inference accuracy on the test set after every n epochs.


In [ ]:
test_interval = summary_params['test_interval']

for epoch in range(1, 1 + optim_params['epochs']):
    # train
    time_beg = time.time()
    net.set_train(True)
    loss_val = []
    for data, in train_dataset:
        step_train_loss = train_step(data)
        loss_val.append(step_train_loss.numpy())
    time_end = time.time()

    print(
        f"epoch: {epoch} train loss: {np.mean(loss_val)} epoch time: {time_end-time_beg:.3f}s")
    if use_scheduler:
        if scheduler_type == "plateau":
            net.set_train(False)
            loss_val = trainer.get_loss(val_dataset)
            scheduler.step(loss_val)
        else:
            scheduler.step()
    # test
    if epoch % test_interval == 0:
        net.set_train(False)
        loss_test = trainer.get_loss(test_dataset)
        print(f'test loss: {loss_test}')


## Result Visualization

The loss descent curve during model training is as follows:

![model](images/loss.png)

It can be observed that after 30,000 training iterations, both the training and test losses have decreased to around 5e-3.

The L2 relative loss predicted by the network for 4 dynamic and 1 algebraic variable is shown in the following figure:
![dynamic variable0 error image](./images/L2relative_error_0.png)
![dynamic variable1 error image](./images/L2relative_error_1.png)
![dynamic variable2 error image](./images/L2relative_error_2.png)
![dynamic variable3 error image](./images/L2relative_error_3.png)
![algebraic variable error image](./images/L2relative_error_4.png)
